# Data Exploration: Vietnamese Sentiment Dataset

**Dataset**: UIT-VSMEC converted to 3 labels  
**Labels**: POSITIVE, NEUTRAL, NEGATIVE  
**Total**: ~4,500 samples

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from wordcloud import WordCloud

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'sans-serif']

## 1. Load Dataset

In [ ]:
# Load all splits
train_df = pd.read_csv('../data/processed_3labels/train.csv', encoding='utf-8')
val_df = pd.read_csv('../data/processed_3labels/val.csv', encoding='utf-8')
test_df = pd.read_csv('../data/processed_3labels/test.csv', encoding='utf-8')

# Combine for overall analysis
df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f"Total samples: {len(df)}")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 2. Label Distribution

In [ ]:
# Label counts
label_counts = df['emotion'].value_counts()
print("Label distribution:")
print(label_counts)
print(f"\nPercentages:")
print(label_counts / len(df) * 100)

In [ ]:
# Visualization: Bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = {'POSITIVE': '#4CAF50', 'NEUTRAL': '#9E9E9E', 'NEGATIVE': '#F44336'}
label_counts.plot(kind='bar', ax=ax1, color=[colors[l] for l in label_counts.index])
ax1.set_title('Label Distribution (Count)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Sentiment', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.tick_params(axis='x', rotation=0)

# Pie chart
label_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%', 
                  colors=[colors[l] for l in label_counts.index],
                  startangle=90)
ax2.set_title('Label Distribution (%)', fontsize=14, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

## 3. Text Length Analysis

In [ ]:
# Calculate text lengths
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print("Text Length Statistics:")
print(df[['text_length', 'word_count']].describe())

In [ ]:
# Visualization: Text length by sentiment
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Character length
for label in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
    data = df[df['emotion'] == label]['text_length']
    ax1.hist(data, alpha=0.6, label=label, bins=30, color=colors[label])
ax1.set_title('Text Length Distribution (Characters)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Characters', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.legend()

# Word count
for label in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
    data = df[df['emotion'] == label]['word_count']
    ax2.hist(data, alpha=0.6, label=label, bins=20, color=colors[label])
ax2.set_title('Word Count Distribution', fontsize=14, fontweight='bold')
ax2.set_xlabel('Words', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Sample Examples

In [ ]:
# Show examples from each class
print("Sample examples:\n")
for label in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
    print(f"\n{'='*60}")
    print(f"{label}:")
    print('='*60)
    samples = df[df['emotion'] == label]['text'].sample(3, random_state=42)
    for i, text in enumerate(samples, 1):
        print(f"{i}. {text}")

## 5. Data Quality Check

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Check for duplicates
duplicates = df.duplicated(subset=['text']).sum()
print(f"\nDuplicate texts: {duplicates}")

# Check for very short texts
very_short = df[df['word_count'] < 3]
print(f"\nTexts with < 3 words: {len(very_short)}")
if len(very_short) > 0:
    print("Examples:")
    print(very_short[['text', 'emotion']].head())

## 6. Split Distribution

In [ ]:
# Check label distribution across splits
print("Train set distribution:")
print(train_df['emotion'].value_counts())
print(f"\nValidation set distribution:")
print(val_df['emotion'].value_counts())
print(f"\nTest set distribution:")
print(test_df['emotion'].value_counts())

In [ ]:
# Visualization: Split comparison
split_data = pd.DataFrame({
    'Train': train_df['emotion'].value_counts(),
    'Val': val_df['emotion'].value_counts(),
    'Test': test_df['emotion'].value_counts()
})

split_data.plot(kind='bar', figsize=(10, 6), 
                color=['#2196F3', '#FF9800', '#9C27B0'])
plt.title('Label Distribution Across Splits', fontsize=14, fontweight='bold')
plt.xlabel('Sentiment', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.legend(title='Split')
plt.tight_layout()
plt.show()